# 14.2. Nuclei segmentation: all fovs

In [58]:

import numpy as np
import pandas as pd
import skimage as ski
from skimage import io, filters
from skimage.filters import threshold_triangle, difference_of_gaussians
from skimage.morphology import disk, extrema
from skimage.measure import regionprops_table
from skimage.segmentation import watershed,relabel_sequential
from scipy import ndimage
from scipy.ndimage import distance_transform_edt
from skimage.color import label2rgb, gray2rgb
from skimage.segmentation import mark_boundaries
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from skimage.feature import peak_local_max

In [59]:
#Apply global variable

GAUSSIAN_SIGMA = 1
DOG_LOW_SIGMA = 3
DOG_HIGH_SIGMA = DOG_LOW_SIGMA * 1.6
DISK_RADIUS = 3
H_MAXIMA_THRESHOLD = 0.5
AREA_MIN = 50

In [60]:
# 1 DAPI image
def count_nuclei_one_image(image_path):  #only use in function
    im = io.imread(image_path).astype(np.float32)
    im_gauss = filters.gaussian(im, sigma=GAUSSIAN_SIGMA)
    im_dog = difference_of_gaussians(im_gauss, low_sigma=DOG_LOW_SIGMA, high_sigma=DOG_HIGH_SIGMA)
    im_threshold = im_dog > threshold_triangle(im_dog)
    im_opened = ndimage.binary_opening(im_threshold, structure=disk(DISK_RADIUS))

    im_dist = np.round(distance_transform_edt(im_opened))
    im_maxima = extrema.h_maxima(im_dist, H_MAXIMA_THRESHOLD)
    im_maxima = ndimage.binary_dilation(im_maxima) & im_opened
    lab_seeds, n = ndimage.label(im_maxima)

    im_water = watershed(-im_dist, markers=lab_seeds, mask=im_opened, watershed_line=True)

    # Filtering by size before plotting
    df = pd.DataFrame(regionprops_table(im_water, properties=['label', 'centroid', 'area']))
    df_filter = df[df['area'] > AREA_MIN]
    im_water = np.where(np.isin(im_water,df_filter['label']), im_water,0)

    im_norm = im / im.max()  # normalize to [0,1]
    im_rgb = gray2rgb(im_norm)
    im_bound = mark_boundaries(im_rgb, im_water, mode='median')
    im_name = image_path.split("registered_")[-1]

    return len(df_filter), im, im_gauss, im_dog, im_threshold, im_opened, im_dist, im_water, im_bound, im_name


## Plotting
def show_image(img, title='', pos=111, cmap='gray'):
    plt.subplot(pos)
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')


def glue_fig(name):
    plt.tight_layout()
    plt.savefig(f"/Users/chann/Downloads/F1 Systems bio/Project/code/Results/Figures/{name}.png", bbox_inches="tight")
    plt.close()


def im_plot(im, im_gauss, im_dog, im_threshold, im_opened, im_dist, im_water, im_bound, im_name):
    fig = plt.figure(figsize=(16, 8))
    fig.suptitle(im_name)

    show_image(im, title='Original image', pos=241)
    show_image(im_gauss, title='Apply Gaussian filter', pos=242, cmap='gray')
    show_image(im_dog, title='Apply DoG filter', pos=243, cmap='gray')
    show_image(im_threshold, title='Apply threshold', pos=244, cmap='gray')
    show_image(im_opened, title='Apply MO', pos=245, cmap='gray')
    show_image(im_dist, title='Apply distance transform', pos=246)
    show_image(label2rgb(im_water, bg_label=0, bg_color='black'), title='Apply watershed', pos=247)
    show_image(im_bound, title='Apply boundary', pos=248, cmap='gray')

    glue_fig(f'fig_overview_workflow{im_name}')

In [61]:
# 4 DAPI images
def cycle_from_path (path):
    return path.split("registered_")[1].split("_")[2]
def four_images(folder, fov_name):
    path_images = sorted(glob.glob(folder + f'out_opt_flow_registered_{fov_name}_c*_DAPI.tif'))
    count_images = []
    count_images_dict = {}
    for path in path_images:
        len_df,im, im_gauss, im_dog, im_threshold, im_opened, im_dist, im_water,im_bound,im_name = count_nuclei_one_image(path)
        im_plot(im, im_gauss, im_dog, im_threshold, im_opened, im_dist, im_water, im_bound, im_name)
        count_images.append(len_df)
        count_images_dict[cycle_from_path(path)] = len_df  #X10_Y10_c01_DAPI.tif => c01
    count_images_dict['mean'] = round(np.mean(count_images))
    count_images_dict['name'] = fov_name
    return count_images_dict

In [62]:
# All of fov

#Take the list of all fov name
import glob
path_fov = glob.glob(PATH + f'out_opt_flow_registered_*_c01_DAPI.tif')

name_fov = [path.split("registered_")[1].split("_c01")[0] for path in path_fov]  #List comprehension
print(len(name_fov))

190


In [63]:
# Count all fov
count_all = []  #list of dict
for name in name_fov:
    fov_result = four_images(PATH, name)
    count_all.append(fov_result)


In [64]:
df_all = pd.DataFrame(count_all)
df_all = df_all[['name', 'c01', 'c02', 'c03', 'c04', 'mean']]
df_all.to_csv('/Users/chann/Downloads/F1 Systems bio/Project/code/Results/Intermediates/Nuclei counting/nuclei_counting_4_DAPI_all_fov.csv', index=False)


1. Flow of code: <br> 1 image -> 4 image in 1 fov  -> all of fov <br>
2. Remember assign correct argument for watershed